## Medical AI
## Skin Lesion Analysis - Towards Melanoma Detection

Skin cancer is the most common cancer globally, with melanoma being the most deadly form. Dermoscopy is a skin imaging modality that has demonstrated improvement for diagnosis of skin cancer compared to unaided visual inspection. However, clinicians should receive adequate training for those improvements to be realized. In order to make expertise more widely available, the International Skin Imaging Collaboration (ISIC) has developed the ISIC Archive, an international repository of dermoscopic images, for both the purposes of clinical training, and for supporting technical research toward automated algorithmic analysis by hosting the ISIC Challenges.



## Preparing the Environment

In [ ]:
#FIXME throughout the notebook search for this tag for things to pay attention to

First, add the dataset that we prepared as input. This prepared dataset is small enough for fast loading, and large enough for good performance.

Click Add Input, and enter this link: https://www.kaggle.com/datasets/matthancaan/medicalai

In [ ]:
#!pip install pytorch-lightning
#!pip install wandb
#from google.colab import drive #for google colab

import os
import glob
import numpy as np
import nibabel as nib
from scipy.ndimage import rotate
import matplotlib.pyplot as plt
import torch
import torchvision
import torchmetrics
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl
import wandb
from pytorch_lightning.loggers import WandbLogger
import cv2

SEED=42

torch.backends.cudnn.deterministic = True
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
np.random.RandomState(SEED)
pl.seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# log in to Weights & Biases (will prompt for your API key: https://wandb.ai/authorize)
wandb.login()


## Classification

In [ ]:
# FIXME set your data directory here
data_dir = '/kaggle/input/datasets/matthancaan/medicalai/classification/'

### Data Modules

In [ ]:
class Scan_Dataset(Dataset):
    def __init__(self, data_dir, transform=False):
      self.transform = transform
      self.data_list = sorted(glob.glob(os.path.join(data_dir,'img*.nii')))

    def __len__(self):
      """defines the size of the dataset (equal to the length of the data_list)"""
      return len(self.data_list)

    def __getitem__(self, idx):
      """ensures each item in data_list is randomly and uniquely assigned an index (idx) so it can be loaded"""
      if torch.is_tensor(idx):
        idx = idx.tolist()

      # loading image
      image_name = self.data_list[idx]
      image = nib.load(image_name).get_fdata()
      
      # setting label from image name
      label = int(image_name.split('.')[0][-1])
      label = torch.tensor(label)

      # apply transforms
      if self.transform:
        image = self.transform(image)

      return image, label

###Visualizing the data

Visualize patients image and labels (cancer or not).

In [ ]:
#@title Visualizing Images { run: 'auto'}
nn_set = 'train' #@param ['train', 'val', 'test']
index  = 0 #@param {type:'slider', min:0, max:355, step:5}

dataset  = Scan_Dataset(os.path.join(data_dir, nn_set))
n_images_display = 5
fig, ax = plt.subplots(1, n_images_display, figsize=(20, 5))
for i in range(n_images_display):
  image, label = dataset[index+i]
  ax[i].imshow(np.uint8(image))
  #ax[i].imshow(np.uint8(np.transpose(image, (1, 2, 0))));
  ax[i].set_title(f'Cancer : {"Yes" if label else "No"}')
  ax[i].axis('off')
plt.suptitle(f'Skin Cancer Images [Indices: {index} - {index + n_images_display} - Images Shape: {dataset[0][0].shape}]');

### Data Augmentation

In [ ]:
class Random_Transform(object):
  """Transform ndarrays in sample."""
  def __init__(self, probability):
    assert isinstance(probability, float) and 0 < probability <= 1, 'Probability must be a float number between 0 and 1'
    self.probability = probability

  def __call__(self, sample):
    if float(torch.rand(1, dtype=torch.float64)) < self.probability:
        # FIXME: your code here if you want to apply augmentation
        sample = sample
    return sample.copy()

### Data Loader

In [ ]:
class Scan_DataModule(pl.LightningDataModule):
  def __init__(self, config):
    super().__init__()
    self.train_data_dir   = config['train_data_dir']
    self.val_data_dir     = config['val_data_dir']
    self.test_data_dir    = config['test_data_dir']
    self.batch_size       = config['batch_size']

    self.train_transforms = transforms.Compose([Random_Transform(0.1), transforms.ToTensor()])
    self.val_transforms  = transforms.Compose([transforms.ToTensor()])

  def setup(self, stage=None):
    self.train_dataset = Scan_Dataset(self.train_data_dir, transform = self.train_transforms)
    self.val_dataset   = Scan_Dataset(self.val_data_dir  , transform = self.val_transforms)
    self.test_dataset = Scan_Dataset(self.test_data_dir  , transform = self.val_transforms)

  def train_dataloader(self):
    return DataLoader(self.train_dataset, batch_size = self.batch_size, num_workers=3)

  def val_dataloader(self):
    return DataLoader(self.val_dataset, batch_size = self.batch_size, num_workers=3)

  def test_dataloader(self):
    return DataLoader(self.test_dataset, batch_size = self.batch_size, num_workers=3)

### CNN - Models

Below you have space for implementing you own architecture and optimizing it the way you want.
A first model is given.

In [ ]:
#FIXME add your models here

class SimpleConvNet(pl.LightningModule):

  def __init__(self):
    super().__init__()

    self.layers = nn.Sequential(
        # conv block 1
        nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1),
        nn.BatchNorm2d(8),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        # conv block 2
        nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(2,2))

    self.classifier = nn.Sequential(
        # linear layers
        nn.AdaptiveAvgPool2d(output_size=(4,4)),
        nn.Flatten(),
        nn.Linear(in_features=4*4*16, out_features=60),
        nn.ReLU(),
        nn.Linear(in_features=60, out_features=1)
    )

  def forward(self, x):
    x = self.layers(x)
    x = self.classifier(x)
    return x


### Optimizers, evaluation metrics and training

Below you should add your optimizer with its hyper-parameters (that should also be optimized), add a loss and some evaluations metrics.

In [ ]:
%cd /kaggle/working
%rm -rf content/models
%mkdir -p content/models
%cd content/models

In [ ]:
models     = {'custom_convnet': SimpleConvNet }

optimizers = {'adam'          : torch.optim.Adam,
              'sgd'           : torch.optim.SGD }

metrics    = {'acc'           : torchmetrics.Accuracy(task="binary").to('cuda'),
              'f1'            : torchmetrics.F1Score(task="binary").to('cuda'),
              'precision'     : torchmetrics.Precision(task="binary").to('cuda'),
              'recall'        : torchmetrics.Recall(task="binary").to('cuda')}

In [ ]:
class Classifier(pl.LightningModule):
  def __init__(self, *args):
    super().__init__()

    # defining model
    self.model_name = config['model_name']
    assert self.model_name in models, f'Model name "{self.model_name}" is not available. List of available names: {list(models.keys())}'
    self.model      = models[self.model_name]().to('cuda')

    # assigning optimizer values
    self.optimizer_name = config['optimizer_name']
    self.lr             = config['optimizer_lr']

  def step(self, batch, nn_set):
    X, y   = batch
    X, y   = X.float().to('cuda'), y.to('cuda')
    y_hat  = self.model(X).squeeze(1)
    y_prob = torch.sigmoid(y_hat)

    loss = F.binary_cross_entropy_with_logits(y_hat, y.float())
    self.log(f'{nn_set}_loss', loss, on_step=False, on_epoch=True)

    for metric_name, metric_fn in metrics.items():
      score = metric_fn(y_prob, y)
      self.log(f'{nn_set}_{metric_name}', score, on_step=False, on_epoch=True)

    return loss

  def training_step(self, batch, batch_idx):
    return {"loss": self.step(batch, "train")}

  def validation_step(self, batch, batch_idx):
    return {"val_loss": self.step(batch, "val")}

  def test_step(self, batch, batch_idx):
    return {"test_loss": self.step(batch, "test")}

  def forward(self, X):
    return self.model(X)

  def configure_optimizers(self):
    assert self.optimizer_name in optimizers, f'Optimizer name "{self.optimizer_name}" is not available. List of available names: {list(models.keys())}'
    return optimizers[self.optimizer_name](self.parameters(), lr = self.lr)

In [ ]:
# FIXME: test dataset now set to validation set, adjust yourself once test data come available
# FIXME: change experiment_name for wandb
config = {
    'train_data_dir' : os.path.join(data_dir, 'train'),
    'val_data_dir'   : os.path.join(data_dir, 'val'),
    'test_data_dir'  : os.path.join(data_dir, 'val'),
    'batch_size'     : 32,
    'optimizer_lr'   : 1e-3,
    'max_epochs'     : 7,
    'model_name'     : 'custom_convnet',
    'optimizer_name' : 'adam',
    'bin'            : 'models/',
    'experiment_name': 'MYLABEL'
}

In [ ]:
data                = Scan_DataModule(config)
classifier          = Classifier(config)
logger              = WandbLogger(project='melanoma-classification', name=config['experiment_name'])
checkpoint_callback = pl.callbacks.ModelCheckpoint(monitor='val_f1', mode = 'max')
trainer             = pl.Trainer(max_epochs=config['max_epochs'],
                                 logger=logger, callbacks=[checkpoint_callback],
                                 default_root_dir=config['bin'], deterministic=False,
                                 log_every_n_steps=1)
trainer.fit(classifier, data)

### Saving the trained model

`ModelCheckpoint` already saved the best epoch to a versioned folder under `config['bin']` (e.g. `models/adam/version_0/checkpoints/...`). That path changes every run, which makes it awkward to find again — especially in a **new session** where that folder won't exist yet.

The cell below copies the best checkpoint to a fixed, predictable path in `/kaggle/working` so it's easy to reload later, either in this same session or a new one.

**Note for reloading in a brand new Kaggle session:** `/kaggle/working` is wiped when a session ends unless you save it — either commit the notebook (Kaggle keeps `/kaggle/working` as version output) or add the `.ckpt` file as a Kaggle Dataset and attach it as input to your next session.

In [ ]:
import shutil

best_model_path = checkpoint_callback.best_model_path
print('Best checkpoint (versioned, session-only):', best_model_path)

# copy to a fixed, predictable path for easy reuse
saved_model_path = '/kaggle/working/best_classifier.ckpt'
shutil.copy(best_model_path, saved_model_path)
print('Saved a stable copy to:', saved_model_path)

If training worked, `logger` printed (or will print) a **wandb** run URL above/below the training progress bar — open it to see live loss/accuracy/F1/precision/recall curves. You can also browse all your runs at [wandb.ai](https://wandb.ai) under the `melanoma-classification` project.

**Testing**

In [ ]:
# point this at your saved checkpoint (fixed path from this session, or a re-attached Kaggle Dataset path)
PATH = saved_model_path  # e.g. '/kaggle/input/<your-dataset>/best_classifier.ckpt' in a new session
# load best model
model = Classifier.load_from_checkpoint(PATH)
model.eval()

# make test dataloader
test_data = Scan_DataModule(config)

# test model (no logger/checkpoint_callback dependency, so this also works in a fresh session)
trainer = pl.Trainer(max_epochs=config['max_epochs'], deterministic=False, log_every_n_steps=1)
trainer.test(model, dataloaders=test_data, verbose=True)

**GRAD CAM (Gradient-weighted Class Activation Mapping) 

In [ ]:
class GradCAMClassification:
    """
    Grad-CAM for binary classification models (output logits [B,1]).
    Hooks into a target convolutional layer (usually the last conv layer
    of the feature extractor, before the classification head).
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove_hooks(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()

    def compute_cam(self, x):
        """
        x: input tensor [1,C,H,W] (already on device, requires_grad not needed beforehand)
        Returns: cam [H,W] normalized to [0,1], and the predicted probability (float)
        """
        self.model.zero_grad()
        x = x.clone().requires_grad_(True)

        logits = self.model(x).squeeze(1)   # [1]
        prob = torch.sigmoid(logits)

        # scalar target: the single output logit ("cancer" class)
        target = logits.sum()
        target.backward()

        grads = self.gradients            # [1,K,h,w]
        acts  = self.activations          # [1,K,h,w]

        weights = grads.mean(dim=(2, 3), keepdim=True)   # GAP over gradients -> [1,K,1,1]
        cam = (weights * acts).sum(dim=1, keepdim=True)  # [1,1,h,w]
        cam = F.relu(cam)

        # upsample to input resolution
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()

        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, prob.item()

### Run Grad-CAM

In [ ]:
model = model.to(device)
model.eval()  # eval mode, but Grad-CAM still needs gradients (do NOT use torch.no_grad!)

# set your layer number and test image number here
layer_number = 3
test_image_number = 0

def get_target_layer(classifier):
    """Returns the last convolutional layer of the underlying feature extractor."""
    inner = classifier.model
    if isinstance(inner, SimpleConvNet):
        return inner.layers[layer_number]          # last Conv2d before the final MaxPool
    else:
        raise ValueError(f'No target layer defined for model type: {type(inner)}')

target_layer = get_target_layer(model)
gradcam = GradCAMClassification(model, target_layer)

# FIXME: test set not defined yet, using validation set instead (see config above)
# grab a sample from the classification test set
test_ds = Scan_Dataset(os.path.join(data_dir, 'val'), transform=transforms.Compose([transforms.ToTensor()]))
image, label = test_ds[test_image_number]

x = image.unsqueeze(0).float().to(device)  # [1,C,H,W]

cam, prob = gradcam.compute_cam(x)
pred_label = int(prob > 0.5)

# original image for overlay (convert to HWC, uint8)
img_np = image.permute(1, 2, 0).cpu().numpy() if image.dim() == 3 else image.cpu().numpy()
img_np = np.uint8(255 * (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8))

heatmap = np.uint8(255 * cam)
heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

overlay = cv2.addWeighted(img_np, 0.6, heatmap_color, 0.4, 0)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(img_np)
ax[0].set_title(f'Input (true: {"Cancer" if label else "No cancer"})')
ax[0].axis('off')
ax[1].imshow(overlay)
ax[1].set_title(f'Grad-CAM overlay (pred: {"Cancer" if pred_label else "No cancer"}, p={prob:.2f})')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## Understanding Grad-CAM

Below, intermediate results can be plotted, if needed, by completing the code.

In [ ]:
acts  = gradcam.activations.detach()            # [1,K,h,w] feature maps of the target layer
grads = gradcam.gradients.detach()              # [1,K,h,w] gradients w.r.t. those feature maps
weights = grads.mean(dim=(2, 3), keepdim=True)  # [1,K,1,1] one weight per channel (GAP over gradients)

raw_cam   = (weights * acts).sum(dim=1, keepdim=True)   # before ReLU, can be negative
relu_cam  = F.relu(raw_cam)                               # after ReLU
upsampled = F.interpolate(relu_cam, size=x.shape[2:], mode='bilinear', align_corners=False)

def normalize01(t):
    t = t.squeeze().cpu().numpy()
    t = t - t.min()
    return t / t.max() if t.max() > 0 else t

n_show = 8
top_channels = weights.squeeze().abs().argsort(descending=True)[:n_show].cpu().numpy()

# FIXME: complete code for plotting here

### Conformal prediction (advanced)

The code below performs conformal prediction for you. It's open ended code to optionally further explore.
Note that the validation set is initially also used as test set, as this is not defined yet.

In [ ]:
# calibration set (val) and evaluation set (test) -- built directly, no Grad-CAM dependency
val_ds  = Scan_Dataset(os.path.join(data_dir, 'val'),  transform=transforms.Compose([transforms.ToTensor()]))
# FIXME test set not defined, setting to validation set
test_ds = Scan_Dataset(os.path.join(data_dir, 'val'), transform=transforms.Compose([transforms.ToTensor()]))

prob_fn = lambda xi: torch.sigmoid(model(xi)).item()  # xi: [1,C,H,W] on `device`


def compute_nonconformity_scores(prob_fn, dataset):
    """prob_fn(x) -> P(class=1) for a single input x [1,C,H,W].
    Nonconformity score for the *true* class: 1 - P(true class)."""
    scores = []
    for idx in range(len(dataset)):
        img_i, label_i = dataset[idx]
        xi = img_i.unsqueeze(0).float().to(device)
        p1 = prob_fn(xi)
        p_true = p1 if int(label_i) == 1 else (1 - p1)
        scores.append(1 - p_true)
    return np.array(scores)


def conformal_threshold(calib_scores, alpha=0.05):
    """Split-conformal quantile of the calibration nonconformity scores.
    Using this as the threshold gives a finite-sample marginal coverage guarantee of
    >= 1 - alpha on exchangeable test data."""
    n = len(calib_scores)
    q_level = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    return np.quantile(calib_scores, q_level, method='higher')


def conformal_prediction_set(p1, threshold):
    """Nonconformity if a candidate class were the true one: 1 - p1 for class 1, p1 for
    class 0. Include the class in the set if its nonconformity is within the threshold."""
    nonconformity = {1: 1 - p1, 0: p1}
    return [c for c, score in nonconformity.items() if score <= threshold]


alpha = 0.05
calib_scores = compute_nonconformity_scores(prob_fn, val_ds)
threshold = conformal_threshold(calib_scores, alpha=alpha)
print(f'Conformal threshold (alpha={alpha}): {threshold:.4f}')

conformal_results = []
for idx in range(len(test_ds)):
    img_i, label_i = test_ds[idx]
    xi = img_i.unsqueeze(0).float().to(device)
    p1 = prob_fn(xi)
    pred_set = conformal_prediction_set(p1, threshold)
    conformal_results.append({'idx': idx, 'label': int(label_i), 'p1': p1,
                               'pred_set': pred_set, 'set_size': len(pred_set),
                               'covered': int(label_i) in pred_set})

coverage      = np.mean([r['covered'] for r in conformal_results])
avg_set_size  = np.mean([r['set_size'] for r in conformal_results])
ambiguous     = [r for r in conformal_results if r['set_size'] != 1]

print(f'Empirical coverage on test set: {coverage:.3f} (target: {1 - alpha:.3f})')
print(f'Average prediction set size: {avg_set_size:.3f} (1.0 = fully confident, 2.0 = "don\'t know")')
print(f'{len(ambiguous)} / {len(conformal_results)} test images got an ambiguous prediction set '
      f'(size != 1) - these are worth a closer manual look.')